# Epic on FHIR Auth Model

Notebook for Epic on FHIR auth model workflows.

In [0]:
%load_ext autoreload
%autoreload 2

In [0]:
import os

In [0]:
# Initialize Databricks Connect when running locally (spark/dbutils are pre-injected on Databricks)
try:
    spark
except NameError:
    from databricks.connect import DatabricksSession
    spark = DatabricksSession.builder.serverless().profile("fe-vm-mkgs-databricks-demos").getOrCreate()
    os.environ["EPIC_ON_FHIR_MLFLOW_USE_PROFILE"] = "1"

In [0]:
# Widget definitions (defaults from bundle variables)
# catalog_use, schema_use, secret_scope_name, client_id_dbs_key, algo, token_url,
# mlflow_experiment_name (deployed experiment), registered_model_name (deployed model)

_catalog_use = "mkgs_dev"
_schema_use = "dev_matthew_giglia_epic_on_fhir"
_secret_scope_name = "epic_on_fhir_oauth_keys"
_client_id_dbs_key = "client_id"
_algo = "RS384"
_token_url = "https://fhir.epic.com/interconnect-fhir-oauth/oauth2/token"
_mlflow_experiment_name = "/Workspace/experiments/[dev matthew_giglia] epic_on_fhir_requests"
_registered_model_name = "mkgs_dev.dev_matthew_giglia_epic_on_fhir.epic_on_fhir_requests"

try:
    dbutils.widgets.text("catalog_use", _catalog_use, "Catalog")
    dbutils.widgets.text("schema_use", _schema_use, "Schema")
    dbutils.widgets.text("secret_scope_name", _secret_scope_name, "Epic Secrets Scope")
    dbutils.widgets.text("client_id_dbs_key", _client_id_dbs_key, "Epic Client ID DB Secrets Key")
    dbutils.widgets.text("algo", _algo, "Epic Token Encryption Algorithm")
    dbutils.widgets.text("token_url", _token_url, "Epic Token URL")
    dbutils.widgets.text("mlflow_experiment_name", _mlflow_experiment_name, "MLflow Experiment Name")
    dbutils.widgets.text("registered_model_name", _registered_model_name, "Registered Model Name")
except Exception:
    pass  # Databricks Connect: widget creation may not be fully supported

In [0]:
# Retrieve widget values (fallback for Databricks Connect)
try:
    catalog_use = dbutils.widgets.get("catalog_use")
    schema_use = dbutils.widgets.get("schema_use")
    secret_scope_name = dbutils.widgets.get("secret_scope_name")
    client_id_dbs_key = dbutils.widgets.get("client_id_dbs_key")
    algo = dbutils.widgets.get("algo")
    token_url = dbutils.widgets.get("token_url")
    mlflow_experiment_name = dbutils.widgets.get("mlflow_experiment_name")
    registered_model_name = dbutils.widgets.get("registered_model_name")
except (AttributeError, Exception):
    catalog_use = _catalog_use
    schema_use = _schema_use
    secret_scope_name = _secret_scope_name
    client_id_dbs_key = _client_id_dbs_key
    algo = _algo
    token_url = _token_url
    mlflow_experiment_name = _mlflow_experiment_name
    registered_model_name = _registered_model_name

In [0]:
import sys
import os

# Get the directory containing this notebook
notebook_dir = os.path.dirname(os.path.abspath('__file__'))

# Add src directory to path if not already present (notebook is already in src/)
if notebook_dir not in sys.path:
	sys.path.append(notebook_dir)

import pandas as pd
import mlflow
from smart_on_fhir.epic_fhir_pyfunc import EpicFhirPyfuncModel

In [0]:
# Only set explicit MLflow URIs when running locally (Databricks Connect).
# On Databricks (notebook/job on cluster or serverless), MLflow is preconfigured.
# Use opt-in env var: set EPIC_ON_FHIR_MLFLOW_USE_PROFILE=1 when running locally.
# DATABRICKS_RUNTIME_VERSION is not reliably set on serverless.

if os.environ.get("EPIC_ON_FHIR_MLFLOW_USE_PROFILE", "").strip() in ("1", "true", "yes"):
    _mlflow_profile = "fe-vm-mkgs-databricks-demos"
    mlflow.set_tracking_uri(f"databricks://{_mlflow_profile}")
    mlflow.set_registry_uri(f"databricks-uc://{_mlflow_profile}")

In [0]:
# Use experiment and registered model from widget defaults.
# The experiment is deployed by the bundle with artifact_location in the mlflow_artifacts volume.
# set_experiment switches to it; if it doesn't exist, creates with default location.
mlflow.set_experiment(mlflow_experiment_name)

In [0]:
# Build the pyfunc model with widget values
model = EpicFhirPyfuncModel(
    secret_scope_name=secret_scope_name,
    client_id_dbs_key=client_id_dbs_key,
    token_url=token_url,
    algo=algo,
)

In [0]:
import json
from pathlib import Path

In [0]:
# Input example for signature: http_method, resource, action, data (optional)
# Row 1: Patient GET | Row 2: Observation create | Row 3: AllergyIntolerance create

import random
from datetime import date, timedelta

def generate_new_payloads():
    # Keep date 2019-09-05, randomly select a valid time for effectiveDateTime
    _obs_date = "2019-09-05"
    _obs_time = f"{random.randint(0, 23):02d}:{random.randint(0, 59):02d}:{random.randint(0, 59):02d}"
    _effective_dt = f"{_obs_date}T{_obs_time}Z"

    # Random date in 2024 for allergy recordedDate (2024 is leap year, 366 days)
    _recorded_date = (date(2024, 1, 1) + timedelta(days=random.randint(0, 365))).isoformat()

    observation_payload = {
        "resourceType": "Observation",
        "status": "final",
        "category": [{"coding": [{"system": "http://hl7.org/fhir/observation-category", "code": "vital-signs", "display": "Vital Signs"}], "text": "Vital Signs"}],
        "code": {"coding": [{"system": "urn:oid:1.2.840.114350.1.13.0.1.7.2.707679", "code": "8", "display": "Heart Rate"}], "text": "Heart Rate"},
        "subject": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "encounter": {"reference": "Encounter/e0u1fd.jUCNqz8ZQuTaMtsQ3"},
        "effectiveDateTime": _effective_dt,
        "valueQuantity": {"value": 75},
    }

    allergy_payload = {
        "resourceType": "AllergyIntolerance",
        "clinicalStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-clinical", "code": "active", "display": "Active"}], "text": "Active"},
        "verificationStatus": {"coding": [{"system": "http://terminology.hl7.org/CodeSystem/allergyintolerance-verification", "code": "unconfirmed", "display": "Unconfirmed"}], "text": "Unconfirmed"},
        "type": "allergy",
        "category": ["medication"],
        "criticality": "low",
        "code": {"coding": [{"system": "http://www.nlm.nih.gov/research/umls/rxnorm", "code": "7980", "display": "Penicillin G"}], "text": "Penicillin"},
        "patient": {"reference": "Patient/T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        "recorder": {"reference": "Practitioner/eM5CWtq15N0WJeuCet5bJlQ3", "display": "Physician Family Medicine, MD"},
        "recordedDate": _recorded_date,
        "reaction": [{"manifestation": [{"coding": [{"system": "http://snomed.info/sct", "code": "247472004", "display": "Hives"}], "text": "Hives"}]}],
    }

    input_example = pd.DataFrame([
        {"http_method": "get", "resource": "Patient", "action": "T1wI5bk8n1YVgvWk9D05BmRV0Pi3ECImNSK8DKyKltsMB"},
        {"http_method": "post", "resource": "Observation", "action": "", "data": json.dumps(observation_payload)},
        {"http_method": "post", "resource": "AllergyIntolerance", "action": "", "data": json.dumps(allergy_payload)},
    ])
    return(input_example)

In [0]:
input_example = generate_new_payloads()

In [0]:
display(input_example)

In [0]:
# Conda env for model serving: JWT (PyJWT), cryptography (for RS384), requests, pandas, mlflow
# conda_env = mlflow.pyfunc.get_default_conda_env()
_conda_env = {
    "name": "epic_on_fhir_serving",
    "channels": ["conda-forge"],
    "dependencies": [
        "python=3.12.3",
        "pip",
        {"pip": ["PyJWT", "cryptography", "requests", "pandas", "mlflow>=3.1", "CloudPickle=3.1.2"]},
    ]
}

In [0]:
# Include smart_on_fhir package source for serving (not pip-installable in serving env)
# MLflow will copy this directory to code/smart_on_fhir in the model artifacts
_smart_on_fhir_path = Path(notebook_dir) / "smart_on_fhir"
_code_path = [str(_smart_on_fhir_path)] if _smart_on_fhir_path.exists() else []

# Infer MLflow signature from model_input (optionally run model.predict(input_example) for output schema)
from mlflow.models import infer_signature

try:
	# Initialize the model's api by calling load_context with a mock context
	model.load_context(None)
	_model_output = model.predict(None, generate_new_payloads())
	_signature = infer_signature(
		input_example
		, pd.DataFrame(
			[
				{
					"response": "string"
				}
			]
		)
	)
except Exception:
	_signature = infer_signature(
		input_example
		, pd.DataFrame(
			[
				{
					 "response": "string"
				}
			]
		)
	)

In [0]:
# _model_output

In [0]:
with mlflow.start_run():
    model_info = mlflow.pyfunc.log_model(
        name="epic_on_fhir_requests",
        python_model=model,
        registered_model_name=registered_model_name,
        input_example=generate_new_payloads(),
        signature=_signature,
        conda_env=_conda_env,
        code_paths=_code_path if _code_path else None,
        params={
            "secret_scope_name": secret_scope_name,
            "client_id_dbs_key": client_id_dbs_key,
            "token_url": token_url,
            "algo": algo
        }
    )
    model_info  # display model_uri, model_id

In [0]:
model_info.registered_model_version

In [0]:
client = mlflow.MlflowClient()

In [0]:
if model_info.registered_model_version == '1': 
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="challenger"
        , version=model_info.registered_model_version
    )
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="champion"
        , version=model_info.registered_model_version
    )
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="prior"
        , version=model_info.registered_model_version
    )
else:
    client.set_registered_model_alias(
        name=registered_model_name
        , alias="challenger"
        , version=model_info.registered_model_version
    )

In [0]:
mlflow.set_active_model(name="epic_on_fhir_requests", model_id=model_info.model_id)

In [0]:
@mlflow.trace
def traced_model_run(model, df):
    logged_model = mlflow.pyfunc.load_model(model.model_uri)
    return(logged_model.predict(df))

In [0]:
for _, row in generate_new_payloads().astype(str).iterrows():
	print(row)
	traced_model_run(model_info, pd.DataFrame([row]))

In [0]:
try:
	# Get current champion model version
	champion_model_version = client.get_model_version_by_alias(
		name=registered_model_name
		, alias="champion"
	)
	print(f"Current champion is {champion_model_version.version}")

	# Try to update 'prior' alias to current champion
	try:
		client.delete_registered_model_alias(
			name=registered_model_name
			, alias="prior"
		)
		print("Removed 'prior' alias")
	except Exception:
		print("No 'prior' alias found.")

	client.set_registered_model_alias(
		name=registered_model_name
		, alias="prior"
		, version=champion_model_version.version
	)
	print(f"Set 'prior' alias to version {champion_model_version.version}")

	# Set new model as champion
	client.set_registered_model_alias(
		name=registered_model_name
		, alias="champion"
		, version=model_info.registered_model_version
	)
	print(f"Set 'champion' alias to version {model_info.registered_model_version}")

except Exception:
	print("No champion model found.")
	client.set_registered_model_alias(
		name=registered_model_name
		, alias="champion"
		, version=model_info.registered_model_version
	)
	print(f"Set 'champion' alias to version {model_info.registered_model_version}")
 
# Delete challenger alias if all successful
client.delete_registered_model_alias(
    name=registered_model_name
    , alias="challenger"
)
print("Deleted 'challenger' alias.")

In [0]:
champion_model_version = client.get_model_version_by_alias(
	name=registered_model_name
	, alias="champion"
)
assert str(champion_model_version.version) == str(model_info.registered_model_version), \
	f"Champion alias version ({champion_model_version.version}) does not match model_info version ({model_info.registered_model_version})"

In [0]:
# Create a simple object with model_uri attribute for traced_model_run
class ModelUriWrapper:
	def __init__(self, model_uri):
		self.model_uri = model_uri

champion_model_uri = f"models:/{champion_model_version.name}/{champion_model_version.version}"
traced_model_run(
	model = ModelUriWrapper(champion_model_uri)
	,df = generate_new_payloads().iloc[[0]]
)